# 아이오닉 5 PDF 소제목 청킹 — 수업 코드 중심

수업의 `8_RAG_파이프라인구축/1_문서로드_청킹.ipynb`에서 배운 흐름을 사용합니다.

**PDF 읽기 → 소제목 내용 선택 → Document 만들기 → 청킹 → Pandas로 확인**

기존 수업과 다른 점은 페이지 전체 대신 **한 가지 기능의 설명을 먼저 골라서** 청킹한다는 것입니다. 위에서 아래로 한 셀씩 실행하세요. 커널은 기존 `study-with-ai`의 Python 가상환경을 선택하세요.

이번에는 PDF 21번째 페이지의 ‘급속 충전기로 충전하기’만 연습합니다. 제목을 자동으로 찾는 코드가 아니라, 원본을 보고 시작·끝 제목을 직접 적는 방식입니다.


## 1. 배운 도구 불러오기
수업에서 사용한 네 가지 도구만 불러옵니다. API 키는 필요하지 않습니다.

In [ ]:
# PDF 파일을 읽는 도구입니다.
from pypdf import PdfReader

# 본문과 출처 정보를 함께 저장하는 도구입니다.
from langchain_core.documents import Document

# 긴 글을 작은 청크로 나누는 도구입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 청킹 결과를 표로 확인하기 위해 Pandas를 불러옵니다.
import pandas as pd


## 2. PDF 읽기
경로는 현재 컴퓨터 기준입니다. 파일 위치가 달라지면 `pdf_path`만 바꾸세요.

In [ ]:
# 읽을 PDF 파일의 위치를 문자열로 저장합니다.
pdf_path = "/Users/anna/Downloads/NE1_2027_ko_KR.pdf"

# PDF를 읽고 reader에 저장합니다.
reader = PdfReader(pdf_path)

# reader.pages에는 PDF의 페이지들이 들어 있습니다. len()으로 개수를 셉니다.
print("전체 페이지 수:", len(reader.pages))


## 3. 한 페이지의 글자 확인하기
PDF 21번째 페이지는 Python에서 20번입니다. PDF 화면의 21번째 페이지에 인쇄된 쪽수는 13쪽입니다.

원본 PDF도 열어 비교하세요. 글자 추출만으로는 그림·표 구조와 절차 번호의 순서까지 보존되지 않습니다.

In [ ]:
# PDF 뷰어 기준으로 확인할 페이지 번호를 정합니다.
target_page = 21

# Python은 0부터 세므로 1을 빼고, extract_text()로 글자를 꺼냅니다.
page_text = reader.pages[target_page - 1].extract_text()

# 글자를 추출하지 못한 경우 None 대신 빈 문자열을 넣습니다.
if page_text is None:
    page_text = ""

# 추출된 글을 출력해 소제목이 어떻게 적혀 있는지 확인합니다.
print(page_text)


## 4. 원하는 소제목의 내용만 고르기

이번에 필요한 문자열 문법은 두 가지입니다.

- `문자열.index("제목")`: 그 제목이 시작하는 문자 위치를 찾습니다.
- `문자열[시작:끝]`: 시작부터 끝 직전까지의 글자를 가져옵니다.

‘급속 충전기로 충전하기’부터 다음 소제목인 ‘휴대용 충전기(ICCB) 사용하기’ 직전까지만 선택합니다. 제목이 없으면 `ValueError`가 납니다. 3번 셀 출력에서 띄어쓰기까지 확인하세요.

이 예제는 설명이 한 페이지 안에서 끝납니다. 여러 페이지에 걸친 소제목은 다음 실습에서 페이지를 연결해 처리합니다.


In [ ]:
# 묶음의 시작 제목입니다. 추출된 원문과 똑같이 적습니다.
title = "급속 충전기로 충전하기"

# 다음 제목은 이 묶음의 끝을 정하는 기준입니다.
next_title = "휴대용 충전기(ICCB) 사용하기"

# 시작 제목의 첫 글자가 있는 위치를 찾습니다.
start = page_text.index(title)

# 시작 제목 뒤에서 다음 제목의 위치를 찾습니다.
end = page_text.index(next_title, start + len(title))

# 두 위치 사이의 글자만 가져옵니다. 다음 제목은 포함되지 않습니다.
raw_section_text = page_text[start:end]

# strip()으로 앞뒤의 불필요한 공백과 줄바꿈을 제거합니다.
section_text = raw_section_text.strip()

# 선택한 내용에 다른 기능의 설명이 섞이지 않았는지 확인합니다.
print(section_text)


## 5. 원본에서 확인한 깨진 제목 정리하기
이 샘플은 경고·주의·알아두기 표기가 깨져 추출됩니다. 원본과 대조해 확인한 세 가지 표기만 복원합니다. 다른 PDF에 그대로 적용하는 공통 규칙은 아닙니다. 절차 번호나 그림의 내용은 이 코드로 복원되지 않습니다.

In [ ]:
# replace()로 깨진 글자를 바꾼 뒤 변수에 다시 저장합니다.
section_text = section_text.replace("҃Ҋ", "[경고]")

# 깨진 주의 표기를 원본에서 확인한 제목으로 바꿉니다.
section_text = section_text.replace("઱੄", "[주의]")

# 깨진 알아두기 표기도 같은 방식으로 바꿉니다.
section_text = section_text.replace("ӝ", "[알아두기]")

# 정리된 내용을 출력합니다. 치환 전 글은 raw_section_text에 남아 있습니다.
print(section_text)


## 6. 수업처럼 Document 만들기
`page_content`는 본문, `metadata`는 본문에 붙이는 출처 정보입니다. 수업의 `pdf_docs` 리스트를 그대로 사용합니다. 지금은 전체 페이지가 아니라 소제목 하나를 넣습니다.

이름이 비슷한 값에 주의하세요. `pdf_page`는 PDF 뷰어 기준 번호이고, `printed_page`는 종이에 인쇄된 쪽수입니다.

In [ ]:
# 이 소제목 묶음을 구별할 이름입니다. 다른 묶음에는 다른 이름을 붙입니다.
parent_id = "fast_charge_p21"

# 작은 청크로 나누기 전의 전체 설명을 Document에 담습니다.
parent_doc = Document(
    page_content=section_text,  # 본문: 정리한 기능 설명 전체
    metadata={
        "source": "NE1_2027_ko_KR.pdf",  # 원본 PDF 파일명
        "pdf_page": target_page,  # PDF 뷰어 기준 21번째 페이지
        "printed_page": "13",  # 이 페이지에 인쇄된 쪽수
        "title": title,  # 소제목
        "parent_id": parent_id  # 나중에 전체 설명을 찾기 위한 연결 이름
    }
)

# 수업에서 사용한 것처럼 빈 리스트를 만듭니다.
pdf_docs = []

# append()로 전체 설명 Document를 리스트에 넣습니다.
pdf_docs.append(parent_doc)

# 리스트의 첫 문서에 들어 있는 출처 정보를 확인합니다.
print(pdf_docs[0].metadata)


## 7. 수업에서 배운 방법으로 청킹하기
수업과 동일하게 `chunk_size=400`, `chunk_overlap=80`으로 시작합니다. 기본 길이 기준은 문자 수입니다. 전체 설명이 길면 여러 청크로 나뉩니다.

겹침은 최대 약 80자를 목표로 하지만, 실제 겹침은 문단 경계에 따라 더 작거나 없을 수 있습니다. 이 숫자는 최적값이 아니라 결과를 비교할 출발점입니다.

In [ ]:
# 긴 글을 나누는 도구를 만듭니다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # 한 청크의 본문을 최대 400자로 나눕니다.
    chunk_overlap=80  # 앞뒤 청크의 내용을 일부 겹치도록 설정합니다.
)

# Document 리스트를 나눕니다. 각 청크에도 metadata가 전달됩니다.
chunks = splitter.split_documents(pdf_docs)

# 만들어진 청크 수를 확인합니다.
print("청크 개수:", len(chunks))


## 8. 기본 for문으로 결과 확인하기
수업에서 배운 `enumerate()`를 사용해 번호와 청크를 하나씩 꺼냅니다. 첫 번호는 0이므로 화면에는 1을 더해 보여줍니다.

In [ ]:
# chunks에서 순서(i)와 문서(chunk)를 하나씩 가져옵니다.
for i, chunk in enumerate(chunks):
    # 청크마다 번호를 붙여 metadata에 저장합니다.
    chunk.metadata["chunk_no"] = i + 1

    # 청크 사이를 구분할 번호를 출력합니다.
    print("청크 번호:", i + 1)

    # 이 청크의 글자 수를 확인합니다.
    print("글자 수:", len(chunk.page_content))

    # 청크의 본문을 출력합니다.
    print(chunk.page_content)

    # 다음 청크와 구분되도록 빈 줄을 출력합니다.
    print()


## 9. Pandas 표로 정리하기
한 청크가 표의 한 행입니다. 리스트 컴프리헨션 대신 `for`문과 `append()`로 차례대로 만듭니다.

In [ ]:
# 표의 각 행을 담을 빈 리스트를 만듭니다.
rows = []

# 청크를 하나씩 꺼내 표에 넣을 정보를 정리합니다.
for chunk in chunks:
    # 딕셔너리의 키가 나중에 표의 열 이름이 됩니다.
    row = {
        "청크번호": chunk.metadata["chunk_no"],  # 청크 순서
        "소제목": chunk.metadata["title"],  # 어떤 기능의 내용인지
        "PDF페이지": chunk.metadata["pdf_page"],  # 원본을 찾을 페이지
        "전체설명ID": chunk.metadata["parent_id"],  # 전체 설명으로 돌아갈 연결
        "글자수": len(chunk.page_content),  # 청크 길이
        "내용": chunk.page_content  # 청크 본문
    }

    # 현재 청크의 정보를 리스트에 한 행씩 추가합니다.
    rows.append(row)

# 딕셔너리 리스트를 Pandas 표로 바꿉니다.
chunks_df = pd.DataFrame(rows)

# 행과 열의 개수를 확인합니다.
print("행·열 수:", chunks_df.shape)

# 열마다 자료형이 무엇인지 확인합니다.
print(chunks_df.dtypes)

# 열별로 빠진 값이 있는지 확인합니다.
print("결측치 개수:")
print(chunks_df.isna().sum())

# 셀 마지막에 표 이름을 적으면 노트북에 표가 표시됩니다.
chunks_df


## 10. 작은 청크의 전체 설명 찾기
작은 청크에는 경고문이 빠져 있을 수 있으므로, 답변에 사용할 때는 연결된 전체 설명도 확인해야 합니다. 아래는 첫 청크가 검색되었다고 가정하는 연습입니다. 아직 질문 검색이나 AI 답변 생성은 하지 않습니다.

In [ ]:
# 첫 번째 청크를 선택합니다. [0]은 첫 항목을 뜻합니다.
selected_chunk = chunks[0]

# 선택한 청크의 전체 설명 ID를 꺼냅니다.
selected_parent_id = selected_chunk.metadata["parent_id"]

# 보관한 전체 설명을 하나씩 확인합니다. 지금은 문서가 한 개 있습니다.
for doc in pdf_docs:
    # 연결 ID가 같으면 선택한 청크가 속한 전체 설명입니다.
    if doc.metadata["parent_id"] == selected_parent_id:
        # 경고·주의를 포함한 전체 설명을 출력합니다.
        print(doc.page_content)

        # 원본의 그림이나 표를 확인할 PDF 페이지 번호도 출력합니다.
        print("원본 확인: PDF", doc.metadata["pdf_page"], "페이지")


## 여기까지 이해하면 됩니다

1. 소제목 하나의 전체 설명을 고릅니다.
2. `Document`에 본문과 출처를 넣습니다.
3. 긴 설명을 `split_documents()`로 나눕니다.
4. 청크의 내용과 전체 설명 연결을 확인합니다.

**직접 해보기:** 7번 셀의 `chunk_size`를 800으로 바꾸고 7~10번 셀을 다시 실행해 청크 개수와 내용을 비교하세요.

이번 버전은 수업 코드에 집중하기 위해 별도 페이지 PDF·JSON 저장, 고급 검사 문법을 생략했습니다. 결과는 `pdf_docs`, `chunks`, `chunks_df` 변수에 있습니다. 커널을 다시 시작하면 위에서부터 재실행하세요.

그림·표는 원본 PDF의 페이지 번호로 찾아갈 수 있습니다. 이미지 추출·표 구조 복원·시각 자료 해석은 아직 구현하지 않았으며 다음 단계에서 추가합니다. 임베딩과 DB 저장도 다음 실습입니다.
